# Lab 4 — Vision-Language Models: CLIP Similarity and a Small VLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humzaahmad906/applied-ml-academy/blob/main/content/vlm-guide/notebooks/13_lab_vlms.ipynb)

Part A uses CLIP to build an image-text similarity explorer (zero-shot classification, similarity matrix). Part B loads SmolVLM to caption an image and answer a visual question.

Runs on Google Colab — for the GPU labs, set Runtime → Change runtime type → GPU.

The full write-up and stack alternatives (open_clip, Ollama, MLX-VLM) live in the [lesson](../13_lab_vlms.md).

In [ ]:
!pip install -q torch transformers accelerate Pillow requests

## Part A — Contrastive Image-Text Similarity with CLIP

### 1. Setup, seeds, sample images

Set seeds for reproducibility, pick a device, and fetch three sample images (with an offline-safe fallback).

In [ ]:
import random
from io import BytesIO

import numpy as np
import requests
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"device: {device}")

# fetch three sample images (Wikimedia, stable URLs)
_URLS = {
    "cat":    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg",
    "dog":    "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg",
    "bridge": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0c/GoldenGateBridge-001.jpg/320px-GoldenGateBridge-001.jpg",
}

def _fetch(url: str) -> Image.Image:
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")

images, img_names = [], []
for name, url in _URLS.items():
    try:
        images.append(_fetch(url))
    except requests.RequestException:
        # solid-color fallback so the rest of the lab still runs offline
        images.append(Image.new("RGB", (224, 224), color=(100, 150, 200)))
    img_names.append(name)
    print(f"  {name}: {images[-1].size}")

### 2. Load CLIP and embed images + captions

CLIP jointly trains a ViT vision encoder and a Transformer text encoder so both modalities land in the same 512-D space: dot product = semantic similarity. L2-normalizing first makes dot product equal cosine similarity.

In [ ]:
clip_id   = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(clip_id)
model     = CLIPModel.from_pretrained(clip_id).to(device)
model.eval()
print(f"CLIP loaded  ({sum(p.numel() for p in model.parameters()) / 1e6:.0f} M params)")

# image features
with torch.inference_mode():
    img_in    = processor(images=images, return_tensors="pt", padding=True).to(device)
    img_feats = model.get_image_features(**img_in)               # [3, 512]
    img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True) # L2-normalize

# caption features
captions = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo of a famous bridge",
    "a photo of a mountain landscape",
    "a painting of a sunset over the ocean",
]

with torch.inference_mode():
    txt_in    = processor(text=captions, return_tensors="pt", padding=True, truncation=True).to(device)
    txt_feats = model.get_text_features(**txt_in)                # [5, 512]
    txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)

### 3. Similarity matrix

The diagonal (matching pairs) should show the highest values. Off-diagonal scores reveal semantic proximity — "cat" and "dog" will be closer to each other than either is to "bridge."

In [ ]:
sim = (img_feats @ txt_feats.T).cpu().numpy()    # [3, 5]

short = [c.replace("a photo of ", "").replace("a painting of ", "")[:18] for c in captions]
print(f"\n{'':>8}  " + "  ".join(f"{s:>18}" for s in short))
for i, name in enumerate(img_names):
    row = "  ".join(f"{sim[i, j]:>18.3f}" for j in range(len(captions)))
    print(f"{name:>8}  {row}")

### 4. Zero-shot classification

No fine-tuning, no training examples — zero-shot works because CLIP saw hundreds of millions of (image, caption) pairs. The logit scale (temperature inverse) sharpens or softens the probability distribution.

In [ ]:
classes = ["a photo of a cat", "a photo of a dog", "a photo of a bridge"]

with torch.inference_mode():
    cls_in    = processor(text=classes, return_tensors="pt", padding=True).to(device)
    cls_feats = model.get_text_features(**cls_in)
    cls_feats = cls_feats / cls_feats.norm(dim=-1, keepdim=True)

# logit_scale is a learned temperature parameter; exp() typically ~100
logit_scale = model.logit_scale.exp()
logits = (logit_scale * img_feats @ cls_feats.T)   # [3, 3]
probs  = logits.softmax(dim=-1).cpu().numpy()

cls_names = ["cat", "dog", "bridge"]
print("\nZero-shot classification:")
for i, name in enumerate(img_names):
    pred    = cls_names[int(probs[i].argmax())]
    bar     = "  ".join(f"{cls_names[j]}: {probs[i, j]:.1%}" for j in range(3))
    correct = "OK" if pred == name else "WRONG"
    print(f"  {name:>8}: {bar}  → {pred} [{correct}]")

## Part B — SmolVLM for Captioning and VQA

### 1. Load the model

SmolVLM (~2 GB) pairs a SigLIP vision encoder with a small SmolLM2 language model via a pixel-shuffle projector.

In [ ]:
from transformers import AutoModelForVision2Seq, AutoProcessor

vlm_id    = "HuggingFaceTB/SmolVLM-Instruct"
print(f"\nLoading {vlm_id} (~2 GB) ...")
vlm_proc  = AutoProcessor.from_pretrained(vlm_id)
vlm_model = AutoModelForVision2Seq.from_pretrained(
    vlm_id,
    torch_dtype=torch.bfloat16,
    _attn_implementation="eager",    # swap to "flash_attention_2" if installed
).to(device)
vlm_model.eval()
print(f"SmolVLM loaded  ({sum(p.numel() for p in vlm_model.parameters()) / 1e6:.0f} M params)")

### 2. Caption and VQA via the processor + generate path

`apply_chat_template` builds the text prompt; the `{"type": "image"}` placeholder marks where the projected visual tokens are inserted, while the actual pixel data is passed separately in the `processor()` call.

In [ ]:
def vlm_ask(image: Image.Image, question: str, max_new_tokens: int = 128) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        }
    ]
    prompt = vlm_proc.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vlm_proc(
        text=prompt, images=[image], return_tensors="pt", padding=True
    ).to(device)

    with torch.inference_mode():
        out_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    n_prompt = inputs["input_ids"].shape[1]        # strip the prompt tokens
    return vlm_proc.decode(out_ids[0, n_prompt:], skip_special_tokens=True).strip()


test_image = images[0]    # cat

print("\n-- Captioning --")
print(vlm_ask(test_image, "Describe this image in one concise sentence."))

print("\n-- Visual Question Answering --")
print(vlm_ask(test_image, "What color is the main animal in this image?"))

### 3. Projector architecture — map to code

`vlm_ask` traces SmolVLM's three stages: (1) the SigLIP vision encoder turns `pixel_values` into patch tokens, (2) a pixel-shuffle + MLP projector compresses and projects them into the language model's embedding space as visual tokens, (3) the SmolLM2 decoder concatenates visual + text tokens and runs standard causal attention. The model never sees raw pixels — only projected patch embeddings. See the lesson for the full breakdown.

## Stacks & alternatives

The lesson also covers three alternative stacks, not run here:

- **open_clip** — for SigLIP, EVA-CLIP, CoCa, MetaCLIP, or checkpoints not on HF Hub; produces identical embeddings to HF CLIP for the same weights.
- **Ollama** (`ollama pull llava:7b`) — zero-boilerplate local serving with an OpenAI-compatible HTTP endpoint, automatic quantization.
- **MLX-VLM** (Apple Silicon) — runs entirely on unified memory for throughput that beats PyTorch-MPS fallbacks; 4-bit SmolVLM stays under 4 GB.

See [13_lab_vlms.md](../13_lab_vlms.md) for the code and tradeoffs of each.